In [0]:
import json
import math

configs = spark.sql("""
SELECT *
FROM configs.migration_config
WHERE enabled = true
""").collect()

all_chunks = []

for c in configs:
    jdbc_url = c["jdbc_url"]
    properties = {
    "user": c["user"],
    "password": c["password"],
    "driver": c["driver"]
}

    strategy = c["load_strategy"]

    # =====================================
    # MANUAL CHUNK
    # =====================================

    if strategy == "MANUAL_CHUNK":

        query = f"""
        (
            SELECT
                MIN({c['chunk_column']}) AS min_id,
                MAX({c['chunk_column']}) AS max_id
            FROM {c['source_table']}
        ) t
        """

        meta = spark.read.jdbc(
            url=jdbc_url,
            table=query,
            properties=properties
        ).collect()[0]

        min_id = meta["min_id"]
        max_id = meta["max_id"]

        current = min_id

        while current <= max_id:

            end = min(
                current + c["chunk_size"] - 1,
                max_id
            )

            all_chunks.append({

                "strategy": "MANUAL_CHUNK",
                "source_name": c["source_name"],

                "source_table": c["source_table"],
                "target_table": c["target_table"],

                "chunk_column": c["chunk_column"],

                "start_id": int(current),
                "end_id": int(end),
                "user": c["user"],
                "password": c["password"],
                "driver": c["driver"],
                "jdbc_url": c["jdbc_url"],
                "partition_column": None,
                "lower_bound": None,
                "upper_bound": None,
                "num_partitions": None

            })
            print("url is ....................")
            print(c["jdbc_url"])

            current = end + 1

    # =====================================
    # JDBC PARTITION
    # =====================================

    elif strategy == "JDBC_PARTITION":

        all_chunks.append({

            "strategy": "JDBC_PARTITION",
            "source_name": c["source_name"],

            "source_table": c["source_table"],
            "target_table": c["target_table"],

            "jdbc_url": c["jdbc_url"],
            "driver": c["driver"],
            "user": c["user"],
            "password": c["password"],

            "chunk_column": None,
            "start_id": None,
            "end_id": None,

            "partition_column": c["partition_column"],
            "lower_bound": c["lower_bound"],
            "upper_bound": c["upper_bound"],
            "num_partitions": c["num_partitions"]

        })

    # =====================================
    # SINGLE
    # =====================================

    else:

        all_chunks.append({

            "strategy": "SINGLE",
            "source_name": c["source_name"],

            "source_table": c["source_table"],
            "target_table": c["target_table"],

            "jdbc_url": c["jdbc_url"],
            "driver": c["driver"],
            "user": c["user"],
            "password": c["password"],

            "chunk_column": None,
            "start_id": None,
            "end_id": None,

            "partition_column": None,
            "lower_bound": None,
            "upper_bound": None,
            "num_partitions": None

        })

print(json.dumps(all_chunks, indent=2))

dbutils.jobs.taskValues.set(
    key="chunks",
    value=json.dumps(all_chunks)
)